In [ ]:
# =========================================================================
# 0. IMPORTS
# =========================================================================
import os, copy, time, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
import seaborn as sns
from PIL import Image
from collections import defaultdict
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    confusion_matrix, roc_curve, auc, precision_recall_curve,
    average_precision_score, accuracy_score
)
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
from scipy import stats

try:
    from skimage.metrics import structural_similarity as ssim
    HAS_SSIM = True
except ImportError:
    HAS_SSIM = False


# =========================================================================
# 1. CONFIGURATION
# =========================================================================
CONFIG = {
    'result_dir':   './results',
    'out_dir':      './results/multi_model',
    'weights_dir':  './results/weights',
    'img_size':     224,
    'batch_size':   8,
    'num_classes':  2,
    'class_names':  ['Insensitive', 'Sensitive'],
    'num_workers':  0,
    'epochs':       10,
    'lr':           1e-4,
    'weight_decay': 1e-2,
    'seed':         42,
    'bootstrap_n':  1000,
    'dpi':          300,
    'device':       'cuda' if torch.cuda.is_available() else 'cpu',
}

for d in [CONFIG['out_dir'], CONFIG['weights_dir']]:
    os.makedirs(d, exist_ok=True)

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])


# =========================================================================
# 2. STYLE
# =========================================================================
def setup_style():
    plt.rcParams.update({
        'font.family':       'Arial',
        'font.size':         10,
        'axes.facecolor':    'white',
        'figure.facecolor':  'white',
        'axes.edgecolor':    '#333333',
        'axes.linewidth':    1.2,
        'axes.grid':         False,
        'xtick.color':       '#333333',
        'ytick.color':       '#333333',
        'text.color':        '#222222',
        'axes.labelcolor':   '#222222',
        'legend.framealpha': 0.9,
        'legend.edgecolor':  '#cccccc',
        'legend.fontsize':   8,
        'pdf.fonttype':      42,
        'ps.fonttype':       42,
    })

setup_style()

PALETTE = [
    '#2166AC','#D6604D','#1A9641','#762A83','#F4A582',
    '#4DAC26','#92C5DE','#E08214','#A6DBA0','#8073AC',
]
CMAP_BLUE = LinearSegmentedColormap.from_list(
    'acad_blue', ['#DEEBF7','#9ECAE1','#3182BD','#08519C'])


# =========================================================================
# 3. I/O HELPERS
# =========================================================================
def out_path(name): return os.path.join(CONFIG['out_dir'], name)

def save_fig(fig, stem):
    fig.savefig(out_path(stem + '.pdf'), format='pdf', bbox_inches='tight', facecolor='white')
    fig.savefig(out_path(stem + '.png'), dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  [fig] {stem}.pdf")

def save_csv(df, stem):
    path = out_path(stem + '.csv')
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f"  [csv] {stem}.csv")


# =========================================================================
# 4. DATASET
# =========================================================================
class MRIDataset(Dataset):
    def __init__(self, df, transform=None):
        self.data = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, int(row['label']), str(row['patient_id']), row['path']


def get_transforms(train=True):
    base = [
        transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]
    if train:
        aug = [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        ]
        return transforms.Compose([base[0]] + aug + base[1:])
    return transforms.Compose(base)


def load_data():
    train_csv = os.path.join(CONFIG['result_dir'], 'train_dataset.csv')
    test_csv  = os.path.join(CONFIG['result_dir'], 'test_dataset.csv')

    if os.path.exists(train_csv) and os.path.exists(test_csv):
        print("Loading existing train/test split...")
        return pd.read_csv(train_csv), pd.read_csv(test_csv)

    print("Splitting dataset.csv into 80% Train, 20% Test...")
    df = pd.read_csv('dataset.csv')
    train_df, test_df = train_test_split(df, test_size=0.2,
                                         random_state=CONFIG['seed'],
                                         stratify=df['label'])
    train_df.to_csv(train_csv, index=False, encoding='utf-8-sig')
    test_df.to_csv(test_csv,   index=False, encoding='utf-8-sig')
    return train_df, test_df


# =========================================================================
# 5. MODEL REGISTRY (only DenseNet121)
# =========================================================================
def build_registry(num_classes):
    """Returns a registry containing only DenseNet121."""
    R = {}

    # DenseNet121
    m = models.densenet121(weights='IMAGENET1K_V1')
    fc_in = 1024
    mid   = fc_in // 2
    m.classifier = nn.Sequential(
        nn.Dropout(0.6),
        nn.Linear(fc_in, mid),
        nn.ReLU(inplace=True),
        nn.Dropout(0.6),
        nn.Linear(mid, num_classes)
    )
    R['DenseNet121'] = dict(
        model=m,
        target_layer=m.features.denseblock4,
        paper='Huang et al., CVPR 2017',
        params_m=8.0
    )
    return R


# =========================================================================
# 6. GRAD-CAM++
# =========================================================================
class UniversalGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = self.gradients = None
        self._disable_inplace(model)
        self._register()

    def _disable_inplace(self, module):
        for child in module.modules():
            if hasattr(child, 'inplace'): child.inplace = False

    def _register(self):
        self.target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'activations', o))
        self.target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradients', go[0]))

    def _to_spatial(self, t):
        if t is None: return None
        if t.ndim == 3:
            B, L, C = t.shape
            H = W = int(L**0.5)
            t = t.permute(0, 2, 1).reshape(B, C, H, W)
        return t if t.ndim == 4 else None

    def __call__(self, x, class_idx=None, method='gradcam++'):
        self.model.eval()
        self.activations = self.gradients = None
        x = x.to(CONFIG['device']).requires_grad_(True)
        out = self.model(x)
        if class_idx is None: class_idx = out.argmax(1).item()
        self.model.zero_grad()
        one_hot = torch.zeros_like(out)
        one_hot[0, class_idx] = 1.
        out.backward(gradient=one_hot, retain_graph=True)

        acts  = self._to_spatial(self.activations)
        grads = self._to_spatial(self.gradients)
        if acts is None or grads is None: return None, class_idx

        if method == 'gradcam++':
            alpha = grads**2 / (2*grads**2 + (grads**3)*acts.sum((2,3), keepdim=True) + 1e-8)
            weights = (alpha * F.relu(grads)).mean((2, 3), keepdim=True)
        else:
            weights = grads.mean((2, 3), keepdim=True)

        cam = F.relu((weights * acts).sum(1, keepdim=True))[0, 0].detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

    def overlay(self, cam, raw_rgb, alpha=0.45):
        h, w = raw_rgb.shape[:2]
        cam_up = cv2.resize(cam, (w, h))
        hm = cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * cam_up), cv2.COLORMAP_JET),
                          cv2.COLOR_BGR2RGB)
        return (raw_rgb * (1 - alpha) + hm * alpha).astype(np.uint8), hm


# =========================================================================
# 7. TRAINING
# =========================================================================
MIXUP_MODELS = {'DenseNet121'}   # DenseNet121 uses Mixup

def train_model(name, model, train_loader, use_mixup=False):
    from sklearn.metrics import roc_auc_score
    device = CONFIG['device']
    model  = model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'],
                            weight_decay=CONFIG['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
    criterion = nn.CrossEntropyLoss(label_smoothing=0.25)

    weight_path = os.path.join(CONFIG['weights_dir'], f'{name}.pth')
    best_auc, history = 0., defaultdict(list)

    if use_mixup:
        print(f"  [{name}] Mixup enabled (alpha=0.4)")

    for epoch in range(CONFIG['epochs']):
        model.train()
        run_loss = correct = total = 0
        t_probs, t_lbls = [], []

        for imgs, labels, _, _ in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()

            if use_mixup:
                lam = np.random.beta(0.4, 0.4)
                idx = torch.randperm(imgs.size(0)).to(device)
                mixed_imgs = lam * imgs + (1 - lam) * imgs[idx]
                out  = model(mixed_imgs)
                loss = lam * criterion(out, labels) + (1 - lam) * criterion(out, labels[idx])
                correct += (out.argmax(1) == labels).sum().item()
            else:
                out  = model(imgs)
                loss = criterion(out, labels)
                correct += (out.argmax(1) == labels).sum().item()

            loss.backward()
            optimizer.step()
            run_loss += loss.item()
            total    += labels.size(0)

            with torch.no_grad():
                t_probs += torch.softmax(out, 1)[:, 1].cpu().tolist()
                t_lbls  += labels.cpu().tolist()

        train_acc = correct / total
        train_auc = roc_auc_score(t_lbls, t_probs) if len(set(t_lbls)) > 1 else 0.5

        history['train_loss'].append(run_loss / len(train_loader))
        history['train_acc'].append(train_acc)
        history['train_auc'].append(train_auc)
        scheduler.step()

        if train_auc >= best_auc:
            best_auc = train_auc
            torch.save(model.state_dict(), weight_path)

        print(f"  [{name}] Ep{epoch+1:02d} "
              f"Loss={history['train_loss'][-1]:.4f} "
              f"TrainAcc={train_acc:.3f} "
              f"TrainAUC={train_auc:.3f}"
              + (' ✓' if train_auc >= best_auc else ''))

    return dict(history), weight_path


# =========================================================================
# 8. EVALUATION
# =========================================================================
def _metrics(y, p, pd_):
    if len(set(y)) < 2:
        return dict(acc=0, sen=0, spe=0, pre=0, f1=0, auc=0.5, ap=0,
                    OR=1, OR_lo=0.1, OR_hi=10)
    cm = confusion_matrix(y, pd_)
    tn, fp, fn, tp = (cm.ravel() if cm.size == 4 else [0, 0, 0, 0])
    acc = (tp+tn) / (tp+tn+fp+fn+1e-8)
    sen = tp / (tp+fn+1e-8);  spe = tn / (tn+fp+1e-8)
    pre = tp / (tp+fp+1e-8);  f1  = 2*pre*sen / (pre+sen+1e-8)
    fpr_, tpr_, _ = roc_curve(y, p);  roc_auc = auc(fpr_, tpr_)
    ap = average_precision_score(y, p)
    tp_, fp_, fn_, tn_ = max(tp, 0.5), max(fp, 0.5), max(fn, 0.5), max(tn, 0.5)
    OR = (tp_*tn_) / (fp_*fn_)
    se_log = np.sqrt(1/tp_ + 1/fp_ + 1/fn_ + 1/tn_)
    OR_lo = np.exp(np.log(OR) - 1.96*se_log)
    OR_hi = np.exp(np.log(OR) + 1.96*se_log)
    return dict(acc=acc, sen=sen, spe=spe, pre=pre, f1=f1, auc=roc_auc, ap=ap,
                OR=OR, OR_lo=OR_lo, OR_hi=OR_hi)


def evaluate_model(name, model, loader, weight_path):
    device = CONFIG['device']
    model  = model.to(device)
    if os.path.exists(weight_path):
        model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
    model.eval()

    y_all, p_all, pd_all, pid_all = [], [], [], []
    with torch.no_grad():
        for imgs, labels, pids, _ in loader:
            out     = model(imgs.to(device))
            p_all  += torch.softmax(out, 1)[:, 1].cpu().tolist()
            pd_all += out.argmax(1).cpu().tolist()
            y_all  += labels.tolist()
            pid_all += list(pids)

    y = np.array(y_all); p = np.array(p_all); pd_ = np.array(pd_all)
    metrics = _metrics(y, p, pd_)

    ci = defaultdict(list)
    for _ in range(CONFIG['bootstrap_n']):
        idx = np.random.choice(len(y), len(y), replace=True)
        if len(set(y[idx])) < 2: continue
        m_b = _metrics(y[idx], p[idx], pd_[idx])
        for k, v in m_b.items(): ci[k].append(v)
    ci = {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in ci.items()}

    fpr_, tpr_, _ = roc_curve(y, p)
    pre_, rec_, _ = precision_recall_curve(y, p)
    return dict(metrics=metrics, ci=ci,
                labels=y, probs=p, preds=pd_, pids=pid_all,
                fpr=fpr_, tpr=tpr_, precision=pre_, recall=rec_)


# =========================================================================
# 9. GRAD-CAM PANEL
# =========================================================================
def plot_gradcam_panel(name, model, target_layer, loader, weight_path,
                       split='test', n_samples=8):
    device = CONFIG['device']
    model  = model.to(device)
    if os.path.exists(weight_path):
        model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
    model.eval()
    cam_gen = UniversalGradCAM(model, target_layer)
    dataset = loader.dataset
    indices = np.random.permutation(len(dataset))[:n_samples*4]

    samples = []
    for idx in indices:
        if len(samples) >= n_samples: break
        img_t, label, pid, img_path = dataset[idx]
        raw_bgr = cv2.imread(img_path)
        if raw_bgr is None: continue
        raw_rgb = cv2.resize(cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB),
                             (CONFIG['img_size'], CONFIG['img_size']))
        try:
            cam, pred = cam_gen(img_t.unsqueeze(0), class_idx=1)
        except Exception: continue
        if cam is None: continue
        ov, _ = cam_gen.overlay(cam, raw_rgb)
        with torch.no_grad():
            prob = torch.softmax(model(img_t.unsqueeze(0).to(device)), 1)[0, 1].item()
        samples.append((raw_rgb, cam, ov, label, pred, prob, pid))

    if not samples:
        print(f"  [{name}] No valid samples for Grad-CAM panel"); return []

    n   = len(samples)
    fig = plt.figure(figsize=(16, n*3.5))
    fig.suptitle(f'{name} — Grad-CAM++ Visualisation Panel ({split})',
                 fontsize=13, fontweight='bold', y=1.01)
    gs = gridspec.GridSpec(n, 4, figure=fig, hspace=0.06, wspace=0.04)
    col_titles = ['Original', 'Grad-CAM++ Heatmap', 'Overlay', 'Edge-Enhanced']

    for row_i, (raw, cam, ov, lbl, pred, prob, pid) in enumerate(samples):
        gray   = cv2.cvtColor(raw, cv2.COLOR_RGB2GRAY)
        cam_up = cv2.resize(cam, (raw.shape[1], raw.shape[0]))
        edge   = cv2.normalize((gray * cam_up).astype(np.float32), None, 0, 255,
                               cv2.NORM_MINMAX).astype(np.uint8)
        edge_rgb  = cv2.cvtColor(edge, cv2.COLOR_GRAY2RGB)
        data_cols = [raw, cam, ov, edge_rgb]
        cmaps     = [None, 'YlOrRd', None, None]

        for col_i, (data, cmap) in enumerate(zip(data_cols, cmaps)):
            ax = fig.add_subplot(gs[row_i, col_i])
            if cmap: ax.imshow(data, cmap=cmap, vmin=0, vmax=1)
            else:    ax.imshow(data)
            ax.axis('off')
            if row_i == 0: ax.set_title(col_titles[col_i], fontsize=9, fontweight='bold')
            if col_i == 0:
                cor = '✓' if lbl == pred else '✗'
                col = '#1A9641' if lbl == pred else '#D6604D'
                ax.set_ylabel(f"{cor} {pid[:10]}\nTrue:{CONFIG['class_names'][lbl]}\n"
                              f"Pred:{CONFIG['class_names'][pred]} ({prob:.2f})",
                              color=col, fontsize=7, rotation=0, labelpad=72, va='center')

    plt.tight_layout()
    stem = f"01_gradcam_panel_{name}_{split}"
    save_fig(fig, stem)
    rows = [dict(pid=pid, true=CONFIG['class_names'][lbl],
                 pred=CONFIG['class_names'][pred], prob=f"{prob:.4f}",
                 correct=(lbl == pred))
            for (_, _, _, lbl, pred, prob, pid) in samples]
    save_csv(pd.DataFrame(rows), stem)
    return samples


# =========================================================================
# 10. EVALUATION VISUALISATIONS (simplified for single model)
# =========================================================================
def plot_confusion_matrix(res, split_name):
    cm = confusion_matrix(res['labels'], res['preds'])
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CONFIG['class_names'],
                yticklabels=CONFIG['class_names'],
                ax=ax, cbar=True, annot_kws={'fontsize': 12})
    ax.set_title(f'DenseNet121 — {split_name} Set\nAUC={res["metrics"]["auc"]:.3f}', fontsize=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout()
    save_fig(fig, f"07_confusion_matrix_{split_name.lower()}")

def plot_roc_single(res, split_name):
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot([0,1],[0,1],'--',color='#aaaaaa',lw=1.2)
    ax.plot(res['fpr'], res['tpr'], color=PALETTE[0], lw=2,
            label=f"DenseNet121 AUC={res['metrics']['auc']:.3f}")
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'ROC Curve — {split_name}')
    ax.legend()
    plt.tight_layout()
    save_fig(fig, f"03_roc_{split_name.lower()}")

def plot_pr_single(res, split_name):
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(res['recall'], res['precision'], color=PALETTE[0], lw=2,
            label=f"DenseNet121 AP={res['metrics']['ap']:.3f}")
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'PR Curve — {split_name}')
    ax.legend()
    plt.tight_layout()
    save_fig(fig, f"04_pr_{split_name.lower()}")

def plot_calibration_single(res, split_name):
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot([0,1],[0,1],'--',color='#aaaaaa')
    try:
        frac, mpred = calibration_curve(res['labels'], res['probs'], n_bins=10)
        ax.plot(mpred, frac, 'o-', color=PALETTE[0], lw=2, ms=5)
    except: pass
    ax.set_xlabel('Mean Predicted Probability'); ax.set_ylabel('Fraction of Positives')
    ax.set_title(f'Calibration — {split_name}')
    plt.tight_layout()
    save_fig(fig, f"09_calibration_{split_name.lower()}")

def plot_score_dist(res, split_name):
    fig, axes = plt.subplots(1,2,figsize=(12,5))
    for ax_i, cls in enumerate(CONFIG['class_names']):
        mask = res['labels'] == ax_i
        probs = res['probs'][mask]
        if len(probs):
            parts = axes[ax_i].violinplot([probs], positions=[0], showmedians=True)
            for pc in parts['bodies']: pc.set_facecolor(PALETTE[0]); pc.set_alpha(0.7)
            axes[ax_i].scatter(np.zeros_like(probs)+np.random.uniform(-.15,.15,size=len(probs)),
                               probs, alpha=0.4, s=12, color=PALETTE[0])
        axes[ax_i].set_title(f'{cls} Samples')
        axes[ax_i].set_ylabel('P(Sensitive)')
        axes[ax_i].set_xticks([0]); axes[ax_i].set_xticklabels(['DenseNet121'])
        axes[ax_i].axhline(0.5, color='#D6604D', ls='--')
    fig.suptitle(f'Score Distribution — {split_name}')
    plt.tight_layout()
    save_fig(fig, f"08_score_dist_{split_name.lower()}")


# =========================================================================
# 11. MAIN PIPELINE (DenseNet121 only)
# =========================================================================
def run():
    print(f"\n{'='*55}")
    print("  DenseNet121 Radiomics Pipeline")
    print(f"  Device: {CONFIG['device']}  |  Output: {CONFIG['out_dir']}")
    print(f"{'='*55}\n")

    train_df, test_df = load_data()
    train_ds = MRIDataset(train_df, get_transforms(train=True))
    test_ds  = MRIDataset(test_df,  get_transforms(train=False))

    train_loader_aug  = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                                   shuffle=True,  num_workers=CONFIG['num_workers'])
    test_loader       = DataLoader(test_ds,  batch_size=1,
                                   shuffle=False, num_workers=CONFIG['num_workers'])
    train_eval_loader = DataLoader(MRIDataset(train_df, get_transforms(False)),
                                   batch_size=1, shuffle=False,
                                   num_workers=CONFIG['num_workers'])

    registry = build_registry(CONFIG['num_classes'])
    name = 'DenseNet121'
    info = registry[name]
    model        = info['model']
    target_layer = info['target_layer']
    weight_path  = os.path.join(CONFIG['weights_dir'], f'{name}.pth')

    # Train if needed
    if os.path.exists(weight_path):
        print(f"  [Skip training] weights found: {weight_path}\n")
    else:
        hist, weight_path = train_model(name, model, train_loader_aug,
                                        use_mixup=(name in MIXUP_MODELS))
        # Optional: save training history plot
        # ...

    # Evaluate on Train and Test
    print("  Evaluating on TRAIN set...")
    train_res = evaluate_model(name, model, train_eval_loader, weight_path)
    print(f"  [Train] AUC={train_res['metrics']['auc']:.4f}  F1={train_res['metrics']['f1']:.4f}")

    print("  Evaluating on TEST set...")
    test_res = evaluate_model(name, model, test_loader, weight_path)
    print(f"  [Test]  AUC={test_res['metrics']['auc']:.4f}  F1={test_res['metrics']['f1']:.4f}")

    # Grad-CAM
    print("  Generating Grad-CAM panels...")
    plot_gradcam_panel(name, model, target_layer, train_eval_loader, weight_path, 'train', 6)
    plot_gradcam_panel(name, model, target_layer, test_loader,       weight_path, 'test',  6)

    # Single‑model visualisations
    print("  Plotting evaluation charts...")
    plot_confusion_matrix(train_res, 'Train')
    plot_confusion_matrix(test_res,  'Test')
    plot_roc_single(train_res, 'Train')
    plot_roc_single(test_res,  'Test')
    plot_pr_single(train_res, 'Train')
    plot_pr_single(test_res,  'Test')
    plot_calibration_single(train_res, 'Train')
    plot_calibration_single(test_res,  'Test')
    plot_score_dist(train_res, 'Train')
    plot_score_dist(test_res,  'Test')

    # Export metrics
    rows = []
    for split, res in [('Train', train_res), ('Test', test_res)]:
        m = res['metrics']
        ci = res['ci']
        row = {'Model': name, 'Split': split}
        for k in ['acc','sen','spe','pre','f1','auc','ap']:
            row[k.upper()] = f"{m[k]:.4f}"
            row[k.upper()+'_CI'] = f"[{ci[k][0]:.4f},{ci[k][1]:.4f}]"
        rows.append(row)
    df_summary = pd.DataFrame(rows)
    save_csv(df_summary, "densenet121_metrics_summary")

    # Export per‑sample predictions
    records = []
    for split, res in [('Train', train_res), ('Test', test_res)]:
        for pid, true, pred, prob in zip(res['pids'], res['labels'], res['preds'], res['probs']):
            records.append(dict(patient_id=pid, true_label=true,
                                true_name=CONFIG['class_names'][true],
                                pred_label=pred, pred_name=CONFIG['class_names'][pred],
                                prob_sensitive=round(prob,6),
                                correct=int(true)==int(pred),
                                split=split))
    df_preds = pd.DataFrame(records).sort_values(['split','patient_id'])
    save_csv(df_preds, "densenet121_predictions")

    print("\n  ✅ DenseNet121 pipeline completed.")


if __name__ == '__main__':
    run()